# Energy Intelligence Platform
## Notebook 01 - RAG Ingestion Pipeline

Este notebook construye la base documental del sistema Energy Intelligence Platform.

Proceso:

1. Lectura de PDFs
2. Extracción de texto
3. División en fragmentos (chunking)
4. Generación de embeddings
5. Creación de índice vectorial FAISS
6. Persistencia en Google Drive

El resultado será utilizado posteriormente por:

- Notebook 02: RAG
- Notebook 03: Sistema Multiagente
- Aplicación Web Lovable

## 1. Instalación

In [1]:
!pip install -q llama-index llama-index-llms-groq

In [2]:
!pip install llama-index-embeddings-huggingface

In [3]:
!pip install faiss-cpu pypdf  sentence-transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 39.2 MB/s eta 0:00:00


In [4]:
!pip install -q langchain langchain-community langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 2. Configuracion

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/EnergyRag")

KNOWLEDGE_DIR = BASE_DIR / "knowledge"

VECTOR_DIR = BASE_DIR / "vector_store"

VECTOR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(KNOWLEDGE_DIR)
print(VECTOR_DIR)

/content/drive/MyDrive/EnergyRag/knowledge
/content/drive/MyDrive/EnergyRag/vector_store


### 2.1 verificacion de documentos

In [7]:
from pathlib import Path

pdf_files = list(
    KNOWLEDGE_DIR.rglob("*.pdf")
)

print(f"PDFs encontrados: {len(pdf_files)}")

for pdf in pdf_files:
    print(pdf.name)

PDFs encontrados: 14
13_PNIEC_España_2024_240924.pdf
14_ise-2025.pdf
15_estrategiaalmacenamiento.pdf
9_google_2025_environmental_report.pdf
10_2025_Microsoft_Environmental_Sustainability_Report.pdf
4_Global_Wind_Report_2026.pdf
5_Global_Market_2025_v1.pdf
1_World_Energy_Outlook_2025.pdf
2_Electricity_2026.pdf
2_Electricity_Market_Report_Update2023.pdf
3_Renewables_2025.pdf
11_Electricity_2024_Analysisandforecastto2026.pdf
12_EnergyandAI.pdf
6_Electricity_Grids_and_Secure_Energy_Transitions.pdf


## 3. Extraccion de documentos

In [8]:
from pypdf import PdfReader
from tqdm import tqdm

documents = []

for pdf_path in tqdm(pdf_files):

    try:
        reader = PdfReader(str(pdf_path))

        text = ""

        for page in reader.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

        documents.append(
            {
                "source": pdf_path.name,
                "path": str(pdf_path),
                "text": text
            }
        )

    except Exception as e:

        print(
            f"ERROR -> {pdf_path.name}: {e}"
        )

print(
    f"Documentos procesados: {len(documents)}"
)

100%|██████████| 14/14 [05:16<00:00, 22.60s/it]

Documentos procesados: 14


### 3.1 Verificacion

In [9]:
for doc in documents[:3]:

    print("=" * 80)

    print(doc["source"])

    print()

    print(doc["text"][:500])

    print()

13_PNIEC_España_2024_240924.pdf

Plan Nacional Integrado 
de Energía y Clima
ACTUALIZACIÓN 2023-2030
Madrid Septiembre 2024
NIPO: 665-20-021-X
Vicepresidencia Tercera del Gobierno de España
Ministerio para la Transición Ecológica y el Reto Demográfico
Autor: Ministerio para la Transición Ecológica y el Reto Demográfico (MITECO)
Edita: Ministerio para la Transición Ecológica y el Reto Demográfico (MITECO)
Revisión de edición: IDAE
Diseño y Maquetación: Abel Guzmán / colectivomelon.com
Imágenes de portada: iStockPhoto y Casey Hor

14_ise-2025.pdf

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
   
 
 
 
Informe del  
Sistema Eléctrico  
202
 5 
Marzo, 2026 
Informe del Sistema Eléctrico. Marzo 2026 
 
 
 
 
 
 
 
 
 
Índice  
1. Presentación........................................................................................................................................................ 1 
2. Resumen ejecutivo ................................................................

### 3.2 Comprobación de tamaño de cada documento

In [10]:
for doc in documents:

    print(
        f"{doc['source']:<60} {len(doc['text']):>10,} caracteres"
    )

13_PNIEC_España_2024_240924.pdf                              2,224,247 caracteres
14_ise-2025.pdf                                                  30,295 caracteres
15_estrategiaalmacenamiento.pdf                                 335,970 caracteres
9_google_2025_environmental_report.pdf                          386,382 caracteres
10_2025_Microsoft_Environmental_Sustainability_Report.pdf       283,542 caracteres
4_Global_Wind_Report_2026.pdf                                   343,170 caracteres
5_Global_Market_2025_v1.pdf                                     322,175 caracteres
1_World_Energy_Outlook_2025.pdf                               1,281,852 caracteres
2_Electricity_2026.pdf                                          537,544 caracteres
2_Electricity_Market_Report_Update2023.pdf                       83,737 caracteres
3_Renewables_2025.pdf                                           562,369 caracteres
11_Electricity_2024_Analysisandforecastto2026.pdf               397,860 caracteres
12_E

## 4. Divición por chunks

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []

for doc in documents:

    category = os.path.basename(
        os.path.dirname(doc["path"])
    )

    parts = splitter.split_text(doc["text"])

    for part in parts:
        chunks.append({
            "source": doc["source"],
            "category": category,
            "text": part
        })

print("Chunks generados:", len(chunks))

Chunks generados: 9986


In [12]:
chunks[0]

{'source': '13_PNIEC_España_2024_240924.pdf',
 'category': 'spain',
 'text': 'Plan Nacional Integrado \nde Energía y Clima\nACTUALIZACIÓN 2023-2030\nMadrid Septiembre 2024\nNIPO: 665-20-021-X\nVicepresidencia Tercera del Gobierno de España\nMinisterio para la Transición Ecológica y el Reto Demográfico\nAutor: Ministerio para la Transición Ecológica y el Reto Demográfico (MITECO)\nEdita: Ministerio para la Transición Ecológica y el Reto Demográfico (MITECO)\nRevisión de edición: IDAE\nDiseño y Maquetación: Abel Guzmán / colectivomelon.com\nImágenes de portada: iStockPhoto y Casey Horner (Unsplash).\nÍNDICE DE \nCONTENIDOS\nRESUMEN EJECUTIVO  ......................................................................................................................................................................................................... 7\n1. PANORAMA DE LA SITUACIÓN ACTUAL  .............................................................................................................

In [13]:
from collections import Counter

Counter(
    [c["category"] for c in chunks]
)

Counter({'spain': 3274,
         'datacenters': 843,
         'renewables': 845,
         'energy': 3104,
         'ai-energy': 1501,
         'grids': 419})

In [14]:
import json

with open(
    "/content/drive/MyDrive/EnergyRag/chunks.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        chunks,
        f,
        ensure_ascii=False
    )

print("Chunks guardados")

Chunks guardados


## 5. Creacion de embeddings

In [15]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [c["text"] for c in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32
)

embeddings = np.array(embeddings)

print("Embeddings shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Embeddings shape: (9986, 384)


### 5.1 Crear Vector Store FAISS

In [16]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vectores indexados:", index.ntotal)

Vectores indexados: 9986


## 6. Guardar indice

In [17]:
import pickle

faiss.write_index(index, "energy_rag.index")

with open("energy_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("FAISS index guardado")
print("Chunks guardados")

FAISS index guardado
Chunks guardados


## 7 Recuperacioón Semántica

In [18]:
#Aquí probamos el RAG antes de meter LLMs
def search(query, k=5):
    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        k
    )

    results = []

    for idx in indices[0]:
        results.append(chunks[idx])

    return results

In [19]:
## primera prueba
results = search(
    "How is AI impacting electricity demand in data centers?"
)

for i, r in enumerate(results):
    print("=" * 80)
    print("RESULT", i + 1)
    print("CATEGORY:", r["category"])
    print("SOURCE:", r["source"])
    print()
    print(r["text"][:1200])

RESULT 1
CATEGORY: ai-energy
SOURCE: 12_EnergyandAI.pdf

slowdown in some efficiency indicators led to faster electricity consumption growth 
* Data starts in 2007. ** Data ends in 2022, estimated for 2022.
Sources: I EA analysis based on data from Cisco ( 2008), Cisco (2015), Cisco ( 2019), ITU ( 2025), Meltwater 
(2024), SPEC (2024), and World Bank (2024a). 
Box 2.1 ⊳ What share of data centre electricity demand comes from AI? 
How much electricity demand comes from AI specifically is a challenging question to 
answer. AI is only one of the workloads that run on data centres , and as AI becomes 
increasingly pervasive, a clear distinction between AI -related and non -AI-related 
workloads becomes more challenging . There is no comprehensive data on the share of 
different kinds of workloads, and service providers or colocation data centre operators 
often have limited visibility over  the specific workloads running in their facilities . 
Moreover, there are often differences in the d